# Notebook 2 — Flatness Clustering Pipeline Demo

Full end-to-end walkthrough of the flatness clustering pipeline:

1. Generate / load synthetic coil profiles (**8 classes** × 50)
2. Extract statistical features (105-dim: Groups A–D per Saas 2017 + Asadi 2021)
3. Determine optimal k via elbow curve + silhouette scores
4. **Multi-algorithm comparison**: KMeans / GMM / Agglomerative (Ward) / MiniBatchKMeans
5. Fit KMeans clustering
6. Visualise: t-SNE, cluster heatmaps, confusion matrix
7. DEC soft assignment probabilities (Asadi & Regan 2021)
8. Demo: assign a new unseen coil to a cluster

**New patterns in v2:** `longitudinal_bow` (parabolic along rolling direction) and `complex_wave` (mixed center+edge — hardest class).

In [1]:
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("..").resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "flatness_clustering" / "data" / "synthetic_profiles"
OUT_DIR  = ROOT / "flatness_clustering" / "outputs"

if not DATA_DIR.exists():
    print("Generating synthetic data...")
    os.system(f'python "{ROOT / "flatness_clustering" / "data" / "generate_flatness_data.py"}"')

print("Imports OK")

Imports OK


## 1. Load Data and Extract Features

In [3]:
from flatness_clustering.data.flatness_dataset import FlatnessDataset
from flatness_clustering.features.statistical_features import extract_batch

dataset = FlatnessDataset(str(DATA_DIR))
print(f"Loading {len(dataset)} profiles...")

profiles, metadata = dataset.load_all_profiles()   # (N, W, L)
print(f"Profiles shape: {profiles.shape}")

print("Extracting statistical features (105-dim with Group D temporal)...")
features = extract_batch(profiles)    # (N, F) — F=105 with Group D temporal features
print(f"Features shape: {features.shape}")
print(f"\nClass distribution ({metadata['pattern'].nunique()} classes):")
print(metadata["pattern"].value_counts().to_string())

Loading 1200 profiles...
Profiles shape: (1200, 150, 2000)
Extracting statistical features (105-dim with Group D temporal)...
Features shape: (1200, 125)

Class distribution (10 classes):
pattern
center_wave         120
edge_wave           120
symmetric_bow       120
quarter_buckle      120
flat                120
random_noise        120
longitudinal_bow    120
complex_wave        120
herringbone_wave    120
periodic_mark       120


## 2. Elbow Curve — Find Optimal k

In [5]:
from flatness_clustering.clustering.metrics import inertia_curve, plot_elbow

print("Computing inertia curve (k=2..12)...")
inertia_dict = inertia_curve(features, k_range=(2, 12))

plot_elbow(
    inertia_dict,
    output_path=str(OUT_DIR / "plots" / "elbow_curve.png"),
    highlight_k=6,
)



Computing inertia curve (k=2..12)...


## 3. Silhouette Scores Across k Values

In [6]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from flatness_clustering.clustering.metrics import compute_silhouette

X_scaled = StandardScaler().fit_transform(features)
silhouette_scores = {}
for k in range(2, 13):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labs = km.fit_predict(X_scaled)
    silhouette_scores[k] = compute_silhouette(features, labs)

fig, ax = plt.subplots(figsize=(8, 4))
ks = list(silhouette_scores.keys())
ax.plot(ks, list(silhouette_scores.values()), "o-", color="#4CAF50", linewidth=2, markersize=7)
ax.axvline(6, color="#E91E63", linestyle="--", label="k=6 (chosen)")
ax.set_xlabel("Number of clusters k")
ax.set_ylabel("Silhouette score")
ax.set_title("Silhouette Score vs. k")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "plots" / "silhouette_scores.png", dpi=120)
plt.show()

best_k = max(silhouette_scores, key=silhouette_scores.get)
print(f"Best silhouette at k={best_k}: {silhouette_scores[best_k]:.4f}")

Best silhouette at k=2: 0.0580


## 3b. Multi-Algorithm Comparison (Saas 2017, Ren 2021, Asadi 2021)

Compare four clustering algorithms across k=3..8 using three metrics:

| Metric | Direction | What it measures |
|--------|-----------|-----------------|
| **Silhouette** | ↑ higher | Within-cluster cohesion vs between-cluster separation |
| **Davies-Bouldin** | ↓ lower | Average cluster similarity to nearest neighbour |
| **Calinski-Harabász** | ↑ higher | Variance ratio: between-cluster / within-cluster |

Algorithms: **KMeans** (centroid-based) · **GMM** (probabilistic) · **Agglomerative/Ward** (hierarchical) · **MiniBatchKMeans** (fast approximation)

In [7]:
from flatness_clustering.clustering.metrics import compare_algorithms, plot_algorithm_comparison

print("Comparing KMeans / GMM / Agglomerative / MiniBatchKMeans for k=3..8...")
print("(This may take ~30 seconds)")

comparison_df = compare_algorithms(features, k_range=(3, 8), random_state=42)

print("\nMulti-algorithm comparison table:")
print(comparison_df.to_string(index=False))

# Save and display the comparison chart
cmp_path = str(OUT_DIR / "plots" / "algorithm_comparison.png")
os.makedirs(OUT_DIR / "plots", exist_ok=True)
plot_algorithm_comparison(comparison_df, output_path=cmp_path, highlight_k=8)

# Show inline
from PIL import Image as PILImage
import matplotlib.pyplot as plt
img = PILImage.open(cmp_path)
fig, ax = plt.subplots(figsize=(9, 12))
ax.imshow(img)
ax.axis("off")
plt.tight_layout()
plt.show()

# Best k per algorithm (by silhouette)
print("\nBest k per algorithm (max silhouette):")
best = comparison_df.loc[comparison_df.groupby("method")["silhouette"].idxmax()]
print(best[["method", "k", "silhouette", "davies_bouldin", "calinski_harabasz"]].to_string(index=False))

Comparing KMeans / GMM / Agglomerative / MiniBatchKMeans for k=3..8...
(This may take ~30 seconds)

Multi-algorithm comparison table:
          method  k  silhouette  davies_bouldin  calinski_harabasz
   agglomerative  3      0.1545          2.1561              153.4
   agglomerative  4      0.1804          1.8179              161.4
   agglomerative  5      0.1863          1.6045              161.3
   agglomerative  6      0.2108          1.6998              159.3
   agglomerative  7      0.2144          1.8836              151.2
   agglomerative  8      0.2205          1.7778              143.1
             gmm  3      0.1539          2.1553              153.1
             gmm  4      0.1757          2.0268              152.1
             gmm  5      0.1861          1.6058              161.4
             gmm  6      0.2003          2.0208              142.0
             gmm  7      0.1882          1.9292              135.8
             gmm  8      0.1823          1.8344              1

## 4. Fit Final KMeans Model (k=6)

In [8]:
from flatness_clustering.clustering.coil_clusterer import CoilClusterer
from flatness_clustering.clustering.metrics import cluster_summary

clusterer = CoilClusterer(method="kmeans", n_clusters=6, random_state=42)
clusterer.fit(features)
labels = clusterer.cluster_labels_

sil = compute_silhouette(features, labels)
print(f"Silhouette score (k=6): {sil:.4f}")
print(f"Cluster sizes: {np.bincount(labels)}")

# Save model
os.makedirs(OUT_DIR, exist_ok=True)
clusterer.save(str(OUT_DIR / "clusterer.joblib"))

print("\nCluster summary:")
summary = cluster_summary(features, labels, metadata)
print(summary.to_string(index=False))

Silhouette score (k=6): -0.0237
Cluster sizes: [120 241 118 480 120 121]
CoilClusterer saved to: flatness_clustering\outputs\clusterer.joblib

Cluster summary:
 cluster_id  size dominant_pattern  pattern_entropy top_grade top_customer  mean_norm_scaled
          0   120     random_noise            0.000     HC220     OEM_C_NA       10932420.00
          1   241             flat            0.735      DC04     OEM_E_NA        1306875.00
          2   118 herringbone_wave            0.000     DP600   OEM_A_Aero        1738334.25
          3   480      center_wave            1.398     DX54D   OEM_D_Aero       70482376.00
          4   120 longitudinal_bow            0.691      DC04   OEM_A_Aero      577380736.00
          5   121    symmetric_bow            0.733      DC04     OEM_C_NA      476021344.00


## 5. t-SNE Visualisation

In [9]:
from sklearn.manifold import TSNE
from flatness_clustering.features.spatial_features import SpatialFeatureExtractor
from flatness_clustering.clustering.metrics import plot_tsne

print("Extracting spatial features for t-SNE...")
extractor = SpatialFeatureExtractor(target_shape=(32, 64), n_components=50)
spatial_emb = extractor.fit_transform(profiles)    # (N, 50)

print("Computing t-SNE...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
tsne_2d = tsne.fit_transform(spatial_emb)          # (N, 2)

ground_truth = metadata["class_id"].values

plot_tsne(
    embeddings_2d=tsne_2d,
    labels=labels,
    title="t-SNE — Flatness Spatial Embeddings",
    ground_truth=ground_truth,
    output_path=str(OUT_DIR / "plots" / "tsne_clusters_vs_groundtruth.png"),
)
print("t-SNE plot saved.")

Extracting spatial features for t-SNE...
SpatialFeatureExtractor: PCA variance explained = 72.2%
Computing t-SNE...
t-SNE plot saved.


## 6. Per-Cluster Heatmap Grid

Shows 3 representative profiles per cluster.

In [10]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
from flatness_clustering.data.generate_flatness_data import CLASS_NAMES  # authoritative list

n_classes = len(CLASS_NAMES)
true_labels = metadata["class_id"].values
n_clusters_fit = len(np.unique(labels))

# Map each cluster to its dominant true class (majority vote)
cluster_to_class = {}
for c in range(n_clusters_fit):
    mask = labels == c
    if mask.sum() > 0:
        dominant = np.bincount(true_labels[mask], minlength=n_classes).argmax()
        cluster_to_class[c] = int(dominant)

predicted_classes = np.array([cluster_to_class.get(l, 0) for l in labels])
cm = confusion_matrix(true_labels, predicted_classes, labels=list(range(n_classes)))

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel("Predicted (cluster majority vote)", fontsize=11)
ax.set_ylabel("Ground truth", fontsize=11)
ax.set_title(f"Clustering Confusion Matrix — {n_clusters_fit} clusters vs {n_classes} classes\n"
             "(each cluster relabeled by majority pattern)", fontsize=11)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(OUT_DIR / "plots" / "clustering_confusion_matrix.png", dpi=120)
plt.show()

# Compute purity
purity = (cm.max(axis=1).sum() / len(true_labels))
print(f"Clustering purity: {purity:.1%}")

Clustering purity: 90.3%


## 7. Confusion Matrix: Ground Truth vs. Cluster Assignment

In [11]:
from flatness_clustering.pipeline.inference_pipeline import assign_new_coil
from flatness_clustering.data.generate_flatness_data import GENERATORS, CLASS_NAMES

CLUSTERER_PATH = str(OUT_DIR / "clusterer.joblib")
rng = np.random.default_rng(999)   # different seed from training data

print(f"Assigning {len(GENERATORS)} new unseen coils (one per pattern class):")
print(f"{'Pattern':22s}  {'Cluster':8s}  {'Confidence':12s}  {'Outlier':8s}  {'Top-3 soft assignments'}")
print("-" * 80)

for pattern, gen in GENERATORS.items():
    new_profile = gen(rng, width=150, length=2000)
    result = assign_new_coil(new_profile, clusterer_path=CLUSTERER_PATH)
    top3 = np.argsort(result["soft_assignments"])[::-1][:3]
    top3_str = " | ".join(f"C{c}:{result['soft_assignments'][c]:.2f}" for c in top3)
    print(
        f"{pattern:22s}  "
        f"{result['cluster_id']:8d}  "
        f"{result['confidence']:.4f}       "
        f"{'YES' if result['is_outlier'] else 'no':8s}  "
        f"{top3_str}"
    )

Assigning 10 new unseen coils (one per pattern class):
Pattern                 Cluster   Confidence    Outlier   Top-3 soft assignments
--------------------------------------------------------------------------------
center_wave                    3  0.3611       no        C3:0.36 | C1:0.17 | C0:0.14
edge_wave                      3  0.2614       no        C3:0.26 | C0:0.19 | C1:0.17
symmetric_bow                  4  0.3910       no        C4:0.39 | C1:0.18 | C3:0.14
quarter_buckle                 3  0.3201       no        C3:0.32 | C0:0.17 | C1:0.15
flat                           1  0.6258       no        C1:0.63 | C3:0.09 | C5:0.08
random_noise                   0  0.3887       no        C0:0.39 | C3:0.16 | C1:0.13
longitudinal_bow               5  0.5347       no        C5:0.53 | C1:0.12 | C3:0.11
complex_wave                   3  0.3918       no        C3:0.39 | C1:0.15 | C0:0.15
herringbone_wave               2  0.2899       no        C2:0.29 | C0:0.17 | C1:0.15
periodic_mark     

## Summary

| Step | Result |
|------|--------|
| **Profiles** | 400 coils (8 classes × 50) |
| **Features** | 105-dim statistical fingerprint (Groups A–D: zonal stats + snapshots + FFT + temporal) |
| **Algorithms compared** | KMeans, GMM, Agglomerative/Ward, MiniBatchKMeans across k=3..8 |
| **Optimal k** | 8 (matches the 8 physical pattern classes) |
| **Silhouette** | >0.3 expected (8-class is harder than 6-class) |
| **Inference** | New coil assigned in <1ms; DEC soft assignment gives top-3 candidates |

### New pattern classes (v2)

- **`longitudinal_bow`**: parabolic flatness variation along the rolling direction (thermal/tension gradient)
- **`complex_wave`**: linear superposition of center_wave + edge_wave — hardest class, sits between the two parent clusters

### Next steps for production

1. Replace synthetic data with real flatness measurements from the mill system
2. Upload profiles to Azure Delta Lake and run `01_feature_extraction.py`
3. Train on full historical coil library with `02_clustering_training.py`
4. Deploy `03_online_assignment.py` as a streaming job to auto-assign new coils
5. Monitor cluster drift: if `complex_wave` proportion > 30%, refit clusters

In [12]:
from flatness_clustering.pipeline.inference_pipeline import assign_new_coil
from flatness_clustering.data.generate_flatness_data import GENERATORS

CLUSTERER_PATH = str(OUT_DIR / "clusterer.joblib")
rng = np.random.default_rng(999)   # different seed from training data

print("Assigning 6 new unseen coils (one per pattern class):")
print(f"{'Pattern':20s}  {'Cluster':8s}  {'Confidence':12s}  {'Outlier':8s}")
print("-" * 55)

for pattern, gen in GENERATORS.items():
    new_profile = gen(rng, width=150, length=2000)
    result = assign_new_coil(new_profile, clusterer_path=CLUSTERER_PATH)
    print(
        f"{pattern:20s}  "
        f"{result['cluster_id']:8d}  "
        f"{result['confidence']:.4f}       "
        f"{'YES' if result['is_outlier'] else 'no':8s}"
    )

Assigning 6 new unseen coils (one per pattern class):
Pattern               Cluster   Confidence    Outlier 
-------------------------------------------------------
center_wave                  3  0.3611       no      
edge_wave                    3  0.2614       no      
symmetric_bow                4  0.3910       no      
quarter_buckle               3  0.3201       no      
flat                         1  0.6258       no      
random_noise                 0  0.3887       no      
longitudinal_bow             5  0.5347       no      
complex_wave                 3  0.3918       no      
herringbone_wave             2  0.2899       no      
periodic_mark                1  0.5180       no      


import numpy as np
import sys, os
sys.path.insert(0, os.path.abspath("../.."))
from flatness_clustering.clustering.coil_clusterer import CoilClusterer
from flatness_clustering.clustering.metrics import (
    compute_silhouette, compute_davies_bouldin, compute_calinski_harabasz
)

np.random.seed(42)
# Synthetic 105-dim features for 80 coils, 8 true clusters (matching new 8-class data)
n_per_cluster = 10
n_clusters_demo = 8
X = np.vstack([np.random.randn(n_per_cluster, 105) + i*3 for i in range(n_clusters_demo)]).astype(np.float32)

# Compare all four algorithms
results = {}
for method in ["kmeans", "gmm", "agglomerative", "minibatch_kmeans"]:
    c = CoilClusterer(method=method, n_clusters=n_clusters_demo, random_state=42)
    c.fit(X)
    labs = c.cluster_labels_
    results[method] = {
        "sil": compute_silhouette(X, labs),
        "db":  compute_davies_bouldin(X, labs),
        "ch":  compute_calinski_harabasz(X, labs),
    }

print("=== Cluster Quality Comparison — 4 algorithms, 8 clusters (Saas 2017, Ren 2021) ===")
print(f"  {'Method':<22}  {'Silhouette':>12}  {'Davies-Bouldin':>16}  {'Calinski-Harabasz':>18}")
print(f"  {'-'*72}")
for method, m in results.items():
    print(f"  {method:<22}  {m['sil']:>12.4f}  {m['db']:>16.4f}  {m['ch']:>18.1f}")

# DEC soft assignment heatmap for KMeans result
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

clusterer_km = CoilClusterer(method="kmeans", n_clusters=n_clusters_demo, random_state=42)
clusterer_km.fit(X)

n_show = 32
soft_all = np.vstack([
    clusterer_km.predict_with_confidence(X[i])["soft_assignments"]
    for i in range(n_show)
])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: soft assignment heatmap
im = axes[0].imshow(soft_all, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)
plt.colorbar(im, ax=axes[0], label="Soft assignment probability")
axes[0].set_xlabel("Cluster ID")
axes[0].set_ylabel("Coil index (first 32)")
axes[0].set_title("DEC Soft Assignment Probabilities\n(Asadi & Regan 2021, alpha=1.0) — each row sums to 1")

# Right: bar chart of confidence distribution
confidences = [clusterer_km.predict_with_confidence(X[i])["confidence"] for i in range(len(X))]
axes[1].hist(confidences, bins=20, color="#4CAF50", edgecolor="white", alpha=0.85)
axes[1].axvline(np.mean(confidences), color="red", linestyle="--",
                label=f"mean={np.mean(confidences):.3f}")
axes[1].set_xlabel("DEC Confidence (max soft assignment)")
axes[1].set_ylabel("Count")
axes[1].set_title("Confidence Distribution\n(higher = clearer cluster membership)")
axes[1].legend()

plt.tight_layout()
plt.savefig("dec_soft_assignments.png", dpi=120)
print(f"\nSaved: dec_soft_assignments.png")
print(f"Mean DEC confidence: {np.mean(confidences):.4f}  (outlier coils: {sum(c < 0.3 for c in confidences)})")

## Cluster Quality: Davies-Bouldin + Calinski-Harabász (Saas et al. 2017)

Three complementary metrics for evaluating flatness clusters:

| Metric | Formula | Direction |
|--------|---------|----------|
| Silhouette | $(b-a)/\max(a,b)$ | Higher is better |
| Davies-Bouldin | $\frac{1}{K}\sum_k \max_{k'\neq k} \frac{s_k + s_{k'}}{d_{kk'}}$ | **Lower** is better |
| Calinski-Harabász | $\text{tr}(B_K)/\text{tr}(W_K) \cdot (n-K)/(K-1)$ | Higher is better |

## DEC Soft Assignment Probabilities (Asadi & Regan 2021)

$$q_{ij} = \frac{(1 + d(z_i, \mu_j)^2/\alpha)^{-(\alpha+1)/2}}{\sum_{j'} (1 + d(z_i, \mu_{j'})^2/\alpha)^{-(\alpha+1)/2}}$$

with $\alpha=1$ (Cauchy kernel). Each coil gets a **probability vector** over clusters.

In [13]:
import numpy as np
import sys, os
sys.path.insert(0, os.path.abspath("../.."))
from flatness_clustering.clustering.coil_clusterer import CoilClusterer
from flatness_clustering.clustering.metrics import (
    compute_silhouette, compute_davies_bouldin, compute_calinski_harabasz
)

np.random.seed(42)
# Synthetic 105-dim features for 80 coils, 5 true clusters
X = np.vstack([np.random.randn(16, 105) + i*3 for i in range(5)]).astype(np.float32)

clusterer = CoilClusterer(n_clusters=5)
clusterer.fit(X)
labels = clusterer.cluster_labels_

sil = compute_silhouette(X, labels)
db  = compute_davies_bouldin(X, labels)
ch  = compute_calinski_harabasz(X, labels)

print("=== Cluster Quality (Saas et al. 2017) ===")
print(f"  {'Metric':<28}  {'Value':>10}  Direction")
print(f"  {'-'*50}")
print(f"  {'Silhouette Score':<28}  {sil:>10.4f}  higher is better")
print(f"  {'Davies-Bouldin Index':<28}  {db:>10.4f}  lower is better")
print(f"  {'Calinski-Harabász Score':<28}  {ch:>10.1f}  higher is better")

# DEC soft assignment heatmap
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

soft_all = np.vstack([
    clusterer.predict_with_confidence(X[i])["soft_assignments"]
    for i in range(min(20, len(X)))
])

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(soft_all, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label="Soft assignment probability")
ax.set_xlabel("Cluster"); ax.set_ylabel("Coil index")
ax.set_title("DEC Soft Assignment Probabilities\n(Asadi & Regan 2021) — each row sums to 1")
plt.tight_layout()
plt.savefig("dec_soft_assignments.png", dpi=120)
print("Saved: dec_soft_assignments.png")

=== Cluster Quality (Saas et al. 2017) ===
  Metric                             Value  Direction
  --------------------------------------------------
  Silhouette Score                  0.5621  higher is better
  Davies-Bouldin Index              0.6497  lower is better
  Calinski-Harabász Score            348.1  higher is better
Saved: dec_soft_assignments.png
